# Modelo Estocástico ACO y Simulación de Montecarlo 📊🐜

Este cuaderno es **completamente autocontenido**. Une la teoría biológica-matemática del comportamiento de las hormigas (basado en los experimentos del doble puente de Deneubourg, 1990 y el libro de Marco Dorigo) con la validación estadística mediante el método de Montecarlo.

---

## 1. El Modelo Estocástico (La Teoría)

Cuando una colonia de hormigas se enfrenta a un puente bifurcado (una rama corta y una rama larga), eligen su camino de forma probabilística. Las hormigas se comunican depositando feromonas.

### La Ecuación de Decisión (Eq. 1.1)
La probabilidad $p_{is}(t)$ de que una hormiga elija la rama corta ($s$) frente a la rama larga ($l$) en el tiempo $t$ se define como:

$$ p_{is}(t) = \frac{(t_s + \varphi_{is}(t))^\alpha}{(t_s + \varphi_{is}(t))^\alpha + (t_s + \varphi_{il}(t))^\alpha} $$

* **$\varphi_{is}(t)$ y $\varphi_{il}(t)$**: Feromonas acumuladas en la rama corta y larga.
* **$t_s$**: Constante de atracción inicial (para evitar divisiones por cero).
* **$\alpha = 2$**: Factor de no linealidad. Elevar al cuadrado hace que el sistema sea extremadamente sensible a pequeñas diferencias en las feromonas.

### El Secreto del Tiempo (Eq. 1.2 y 1.3)
¿Por qué gana la rama corta si no hay evaporación de feromonas? Por el **retraso (delay)**:

$$ d\varphi_{is}/dt = \psi p_{js}(t - t_s) + \psi p_{is}(t) $$
$$ d\varphi_{il}/dt = \psi p_{jl}(t - r \cdot t_s) + \psi p_{il}(t) $$

Las hormigas que toman la rama corta regresan en el tiempo $t_s$, mientras que las de la rama larga regresan en $r \cdot t_s$ (donde $r>1$). Al regresar más rápido, las de la rama corta depositan feromonas antes, inclinando la balanza probabilística a su favor.

## 2. Uniendo la Teoría con Montecarlo

Dado que las decisiones de las hormigas están basadas en el azar (modelo estocástico), correr el experimento una sola vez no demuestra nada estadísticamente. Aquí es donde entra **Montecarlo**.

El método de Montecarlo consiste en **repetir la simulación desde cero miles de veces** para observar el comportamiento promedio de la colonia. En la **Figura 1.4** del libro de Dorigo, se grafican los resultados de 1,000 simulaciones.

### ¿Cómo interpretar los Histogramas?
* **Eje X (% de tráfico):** Representa qué proporción de la colonia eligió un camino específico al final de una simulación.
* **Eje Y (% de experimentos):** De las 1,000 simulaciones, cuántas terminaron con ese resultado.

**Resultados Esperados:**
1. **Ramas iguales ($r=1$):** La gráfica formará una "U". La colonia casi nunca se divide 50/50. Por el efecto de refuerzo positivo, siempre convergen hacia un solo camino al azar (a veces el A, a veces el B).
2. **Rama larga es el doble ($r=2$):** La gráfica se amontona en la derecha (80-100%). Esto prueba estadísticamente que en la enorme mayoría de los escenarios posibles, la rama corta inunda el sistema de feromonas más rápido y atrae a toda la colonia.

---
## 3. Implementación Algorítmica y Simulación Interactiva

A continuación, ejecutaremos 1,000 colonias completas en milisegundos. Usa los deslizadores (sliders) para configurar los parámetros y luego presiona **Ejecutar Simulación** para replicar los gráficos.

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [7]:
def experimento_individual(ratio_r, alpha):
    """Simula un experimento completo (1000 hormigas cruzando) de forma optimizada"""
    t_s = 20.0  # Tiempo base para cruzar la rama corta
    t_l = t_s * ratio_r  # Tiempo para la rama larga
    phi_s = 0.0
    phi_l = 0.0
    
    # Listas para guardar en qué momento (t) se depositará la feromona
    eventos_s = []
    eventos_l = []
    
    hormigas_rama_corta = 0
    
    # Simular 1000 hormigas (llegando 1 cada 2 segundos, psi=0.5)
    for i in range(1, 1001):
        t_actual = i * 2.0
        
        # 1. Aplicar las feromonas de las hormigas que YA cruzaron (t <= t_actual)
        while eventos_s and eventos_s[0] <= t_actual:
            phi_s += 1.0
            eventos_s.pop(0)
            
        while eventos_l and eventos_l[0] <= t_actual:
            phi_l += 1.0
            eventos_l.pop(0)
            
        # 2. Decisión usando la Ecuación 1.1
        num_s = (t_s + phi_s) ** alpha
        num_l = (t_s + phi_l) ** alpha
        p_s = num_s / (num_s + num_l)
        
        # 3. La hormiga toma su decisión estocástica
        if np.random.rand() < p_s:
            # Eligió corta: depositará feromona cuando termine de cruzar
            eventos_s.append(t_actual + t_s)
            # Solo contamos el tráfico de la hormiga 501 a la 1000 (estabilización)
            if i > 500: 
                hormigas_rama_corta += 1
        else:
            # Eligió larga
            eventos_l.append(t_actual + t_l)
            
    # Retornar porcentaje de tráfico en rama corta de las últimas 500 hormigas
    return (hormigas_rama_corta / 500.0) * 100.0


def montecarlo_aco(num_experimentos, ratio_r, alpha):
    """Ejecuta N simulaciones y grafica el histograma"""
    
    # Ejecutar Montecarlo masivo
    resultados = [experimento_individual(ratio_r, alpha) for _ in range(num_experimentos)]
    
    # Agrupar los resultados en los 5 rangos de la gráfica (0-20, 20-40, etc.)
    bins = [0, 20, 40, 60, 80, 100]
    conteos, _ = np.histogram(resultados, bins=bins)
    porcentajes = (conteos / num_experimentos) * 100.0
    
    # --- DIBUJO DE LA GRÁFICA ---
    plt.figure(figsize=(8, 5))
    etiquetas = ['0-20', '20-40', '40-60', '60-80', '80-100']
    
    plt.bar(etiquetas, porcentajes, color='lightgray', edgecolor='black', width=0.6)
    
    # Construcción segura del título para evitar errores de caracteres (como '\a')
    simb_alpha = r"$\alpha$"
    titulo_1 = f"Resultados de {num_experimentos} simulaciones Montecarlo"
    titulo_2 = f"$r$={ratio_r}  |  {simb_alpha}={alpha}"
    plt.title(f"{titulo_1}\n{titulo_2}", fontsize=14, pad=15)
    
    plt.xlabel('% de tráfico en la rama corta', fontsize=12)
    plt.ylabel('% de experimentos', fontsize=12)
    plt.ylim(0, max(100, max(porcentajes) + 10))
    
    # Mostrar números exactos sobre las barras
    for i, v in enumerate(porcentajes):
        plt.text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')
        
    # Estética de gráfica académica
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    
    plt.show()

# --- INTERFAZ GRÁFICA CON BOTÓN ---
w_experimentos = widgets.IntSlider(min=100, max=2500, step=100, value=1000, description='Simulaciones:')
w_ratio = widgets.FloatSlider(min=1.0, max=3.0, step=0.1, value=1.0, description='Ratio Long. (r):')
w_alpha = widgets.FloatSlider(min=0.5, max=3.0, step=0.5, value=2.0, description='Alpha ($\alpha$):')
btn_ejecutar = widgets.Button(description='Ejecutar Simulación', button_style='success', icon='play')
salida = widgets.Output()

def ejecutar_simulacion(b):
    with salida:
        salida.clear_output(wait=True)  # Limpia la gráfica anterior antes de dibujar la nueva
        montecarlo_aco(w_experimentos.value, w_ratio.value, w_alpha.value)

# Conectar el botón a la función
btn_ejecutar.on_click(ejecutar_simulacion)

# Mostrar la interfaz en pantalla
display(w_experimentos, w_ratio, w_alpha, btn_ejecutar, salida)

# Ejecutar la simulación una vez automáticamente al inicio
ejecutar_simulacion(None)

IntSlider(value=1000, description='Simulaciones:', max=2500, min=100, step=100)

FloatSlider(value=1.0, description='Ratio Long. (r):', max=3.0, min=1.0)

FloatSlider(value=2.0, description='Alpha ($\x07lpha$):', max=3.0, min=0.5, step=0.5)

Button(button_style='success', description='Ejecutar Simulación', icon='play', style=ButtonStyle())

Output()